In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama

In [3]:
llm = ChatOllama(
    model="llama3.2",
    temperature=0,
    base_url="http://172.31.0.1:11434"
)

subgraph_llm = ChatOllama(
    model="llama3.2",
    temperature=0,
    base_url="http://172.31.0.1:11434"
)

In [4]:
class Substate(TypedDict):
    input_text: str
    translated_text: str

In [5]:
def translate_text(state: Substate):

    prompt = f"""
    Translate the following text to Hindi.
    Keep it natural and do not add extra content:

    {state['input_text']}
    """.strip()

    translated = subgraph_llm.invoke(prompt).content

    return {
        "translated_text": translated
    }

In [6]:
subgraph = StateGraph(Substate)

subgraph.add_node(
    "translate_text",
    translate_text
)

subgraph.add_edge(
    START,
    "translate_text"
)

subgraph.add_edge(
    "translate_text",
    END
)

subgraph_chat = subgraph.compile()

In [7]:
class ParentState(TypedDict):
    question: str
    answer_eng: str
    input_text: str
    translated_text: str

In [8]:
def generate_answer(state: ParentState):

    answer = llm.invoke(
        f"""
        Answer the following question in English.
        Keep the answer within 100 words.

        Question: {state['question']}
        """.strip()
    )

    return {
        "answer_eng": answer.content,
        "input_text": answer.content
    }

In [9]:
parent = StateGraph(ParentState)

parent.add_node(
    "generate_answer",
    generate_answer
)

parent.add_node(
    "translate",
    subgraph_chat
)

parent.add_edge(
    START,
    "generate_answer"
)

parent.add_edge(
    "generate_answer",
    "translate"
)

parent.add_edge(
    "translate",
    END
)

parent_chat = parent.compile()

In [10]:
result = parent_chat.invoke({
    "question": "What quantum physics is and how it works in 100 words only?"
})

English Answer:
Quantum physics, also known as quantum mechanics, is a branch of physics that studies the behavior of matter and energy at an atomic and subatomic level. It describes the interactions between particles and their properties, such as wave-particle duality, superposition, and entanglement. Quantum physics explains how particles can exist in multiple states simultaneously and be connected across vast distances. It's a probabilistic theory, meaning that predictions are made based on probabilities rather than definite outcomes. This field has led to numerous breakthroughs in technology, including transistors, lasers, and computer chips. Its principles govern the behavior of matter at the smallest scales.

Hindi Answer:
अणु और सूक्ष्म अणुओं के स्तर पर पदार्थ और ऊर्जा की व्यवहार को अध्ययन करने वाली भौतिकी की शाखा है, जिसे क्वांटम फिजिक्स या क्वांटम मैकेनिक्स भी कहा जाता है। यह परत्व-भाग-स्तरीय दuality, सुपरपोज़िशन और एक्सेंट्रेलिटी जैसी गुणों के बीच के संवाद को अध्ययन करती है। 